# 🎯 Customer Segmentation: RFM Rule Engine & K-Means Clustering
---
### **Analytical Framework & Objectives**
*Prepared strictly in accordance with Sections 4 & 5 of the Retail Customer Analytics Project Guide*

In this notebook, we transform transaction-level data into an **Analytical Customer Base Table (ABT)** and implement two distinct customer segmentation methodologies:
1. **The 7-Tier Hierarchical RFM Rule Engine**: An interpretable, business-driven rule hierarchy that categorizes every customer into an operational persona (*Champions, Loyal, At-Risk Valuable, New, Promising, Hibernating, Developing*).
2. **Unsupervised Machine Learning (K-Means + PCA)**: Data-driven clustering with mathematical log-transformations, standardization, Elbow Method ($WCSS$), Silhouette validation, and Principal Component Analysis (PCA) projection.
3. **Comparative Evaluation**: Cross-tabulating rule-based personas against ML clusters to assess interpretability, stability, and business actionability.

---
### **Workflow Checklist:**
- [x] **Step 1**: Load & Reconstruct Validated Purchase View
- [x] **Step 2**: Calculate Customer-Level RFM Metrics & Behavioral Attributes
- [x] **Step 3**: Score RFM Dimensions (1 to 5) Handling Quantile Ties
- [x] **Step 4**: Implement the Exact 7-Tier Hierarchical Rule Engine
- [x] **Step 5**: Profile & Visualize the 7 Customer Personas
- [x] **Step 6**: Mathematical ML Preparation (Log-Transform & Scaling)
- [x] **Step 7**: K-Means Hyperparameter Optimization (Elbow & Silhouette)
- [x] **Step 8**: Train Winning K-Means Model & 2D PCA Cluster Projection
- [x] **Step 9**: Rule-Based vs. K-Means Comparative Cross-Tabulation
- [x] **Step 10**: Export Segmented Analytical Table for Retention Modeling


> 🧠 **ANALYSIS TYPE**: Setup & Data Science Library Imports  
> 🎯 **WHAT THIS CODE ACTUALLY DOES**:  
> 1. Imports `pandas`, `numpy`, `matplotlib`, and `seaborn`.  
> 2. Imports Scikit-Learn modules: `StandardScaler` for normalization, `KMeans` for unsupervised clustering, `silhouette_score` for cluster validation, and `PCA` for 2D dimensionality reduction.  
> 3. Configures global chart aesthetics.  
> 🔍 **WHAT TO OBSERVE**: Output confirms all ML modules are ready.


In [ ]:
# Setup: Essential Data Science & Machine Learning Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt

# Scikit-Learn ML modules
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

import warnings
warnings.filterwarnings('ignore')

# Visual style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.titleweight'] = 'bold'

print('Machine learning and clustering environment initialized successfully!')


## 📦 Step 1: Load and Filter Qualifying Purchase Transactions
---
> 🧠 **ANALYSIS TYPE**: Data Ingestion & Scope Enforcement  
> 🎯 **WHAT THIS CODE ACTUALLY DOES**:  
> 1. Loads raw `online_retail.csv`.  
> 2. Converts `InvoiceDate` to datetime.  
> 3. Filters to known customers (`CustomerID.notnull()`).  
> 4. Enforces the **Purchase View** criteria: `Quantity > 0`, `UnitPrice > 0`, non-cancellation (no 'C' prefix), and removes administrative stock codes (`POST`, `D`, `M`, `BANK CHARGES`).  
> 5. Calculates `LineValue = Quantity * UnitPrice`.  
> 🔍 **WHAT TO OBSERVE**: Total qualifying purchase rows (~392,000) and number of unique customers (~4,338).


In [ ]:
# Load raw transactions and construct qualifying purchase view
df_raw = pd.read_csv('data/online_retail.csv', encoding='latin1')
df_raw['InvoiceDate'] = pd.to_datetime(df_raw['InvoiceDate'])

admin_codes = ['POST', 'D', 'M', 'BANK CHARGES', 'PADS', 'DOT', 'CRUK']

# Qualifying Purchase View with known CustomerID
df_purchases = df_raw[
    (df_raw['CustomerID'].notnull()) &
    (~df_raw['InvoiceNo'].astype(str).str.startswith('C')) &
    (df_raw['Quantity'] > 0) &
    (df_raw['UnitPrice'] > 0) &
    (~df_raw['StockCode'].astype(str).isin(admin_codes))
].copy()

df_purchases['CustomerID'] = df_purchases['CustomerID'].astype(int).astype(str)
df_purchases['LineValue'] = df_purchases['Quantity'] * df_purchases['UnitPrice']

print(f'Qualifying Purchase Transactions: {len(df_purchases):,}')
print(f'Unique Known Customers:          {df_purchases["CustomerID"].nunique():,}')
print(f'Total Gross Purchase Value:      £{df_purchases["LineValue"].sum():,.2f}')
df_purchases.head()


## 🔢 Step 2: Compute Customer-Level RFM & Behavioral Features
---
> 🧠 **ANALYSIS TYPE**: Analytical Base Table (ABT) Feature Engineering  
> 🎯 **WHAT THIS CODE ACTUALLY DOES**:  
> 1. Defines the **Reference / Snapshot Date**: exactly one day after the maximum observed transaction (`2011-12-10`).  
> 2. Aggregates data by `CustomerID` to engineer:  
>    - **Recency ($R$)**: Days between snapshot date and customer's latest purchase.  
>    - **Frequency ($F$)**: Total distinct purchase invoices (`InvoiceNo.nunique()`).  
>    - **Monetary ($M$)**: Total gross qualifying spend (`LineValue.sum()`).  
>    - **AvgOrderValue ($AOV$)**: Total spend divided by total orders.  
>    - **CustomerTenureDays**: Days between customer's first purchase and snapshot date.  
> 🔍 **WHAT TO OBSERVE**: Each row in `customer_df` represents one unique customer persona.


In [ ]:
# Establish snapshot reference date
snapshot_date = df_purchases['InvoiceDate'].max() + dt.timedelta(days=1)
print(f'Analysis Snapshot Reference Date: {snapshot_date.strftime("%Y-%m-%d")}')

# Aggregate at Customer Grain
customer_df = df_purchases.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency=('InvoiceNo', 'nunique'),
    Monetary=('LineValue', 'sum'),
    FirstPurchase=('InvoiceDate', 'min'),
    DistinctSKUs=('StockCode', 'nunique')
).reset_index()

customer_df['CustomerTenureDays'] = (snapshot_date - customer_df['FirstPurchase']).dt.days
customer_df['AvgOrderValue'] = (customer_df['Monetary'] / customer_df['Frequency']).round(2)
customer_df.drop(columns=['FirstPurchase'], inplace=True)

print(f'Customer Feature Table Shape: {customer_df.shape[0]} customers, {customer_df.shape[1]} features')
display(customer_df.describe().round(2))
customer_df.head()


## 📊 Step 3: Score RFM Dimensions (1 to 5) Handling Quantile Ties
---
> 🧠 **ANALYSIS TYPE**: Quantile Scoring & Bounded Ranking  
> 🎯 **WHAT THIS CODE ACTUALLY DOES**:  
> 1. Scores **Recency (R)**: Customers with lowest recency get 5 (most recent), highest recency get 1.  
> 2. Scores **Frequency (F)**: Because ~34% of customers bought only once (`Frequency = 1`), standard quantile binning would collapse with duplicate bin edges! We use `pd.qcut(rank(method='first'))` or explicit thresholds to ensure mathematically sound 1 to 5 scores.  
> 3. Scores **Monetary (M)**: Top 20% spenders get 5, bottom 20% get 1.  
> 🔍 **WHAT TO OBSERVE**: Verify score distributions (1, 2, 3, 4, 5) across R_Score, F_Score, and M_Score.


In [ ]:
# Recency Scoring: Lower days = Higher Score (5 is best)
customer_df['R_Score'] = pd.qcut(customer_df['Recency'], q=5, labels=[5, 4, 3, 2, 1]).astype(int)

# Frequency Scoring: Handle heavy tie-concentration at Frequency = 1 using rank method
customer_df['F_Score'] = pd.qcut(customer_df['Frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]).astype(int)

# Monetary Scoring: Higher spend = Higher Score (5 is best)
customer_df['M_Score'] = pd.qcut(customer_df['Monetary'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]).astype(int)

# Combined RFM Score string (e.g. '555' for top champions)
customer_df['RFM_Segment_Code'] = customer_df['R_Score'].astype(str) + customer_df['F_Score'].astype(str) + customer_df['M_Score'].astype(str)

print('=== RFM SCORE DISTRIBUTION CHECK ===')
print('R_Score value counts:\n', customer_df['R_Score'].value_counts().sort_index())
print('\nF_Score value counts:\n', customer_df['F_Score'].value_counts().sort_index())
print('\nM_Score value counts:\n', customer_df['M_Score'].value_counts().sort_index())
customer_df.head()


## 👑 Step 4: Implement the Exact 7-Tier Hierarchical Rule Engine
---
> 🧠 **ANALYSIS TYPE**: Hierarchical Expert Rule Engine  
> 🎯 **WHAT THIS CODE ACTUALLY DOES**:  
> Implements the exact top-to-bottom assignment rules defined in Section 4 of the Guide:
> 1. **New Customers**: `Frequency == 1` and `R_Score >= 4`
> 2. **Champions**: `R_Score >= 4` and `F_Score >= 4` and `M_Score >= 4`
> 3. **Loyal Customers**: `R_Score >= 3` and `F_Score >= 4`
> 4. **At Risk Valuable**: `R_Score <= 2` and (`F_Score >= 4` or `M_Score >= 4`)
> 5. **Promising Customers**: `R_Score >= 4`
> 6. **Hibernating Customers**: `R_Score <= 2`
> 7. **Developing Customers**: All remaining customers  
> 🔍 **WHAT TO OBSERVE**: Checks that every single customer receives exactly one segment label without gaps or contradictions.


In [ ]:
# Define the 7-tier rule function strictly matching the project guide
def assign_rfm_segment(row):
    r = row['R_Score']
    f = row['F_Score']
    m = row['M_Score']
    freq = row['Frequency']
    
    # Rule 1: New customers (first-time recent buyers)
    if freq == 1 and r >= 4:
        return 'New Customers'
    
    # Rule 2: Champions (bought recently, buy often, spend the most)
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'
    
    # Rule 3: Loyal customers (buy regularly and responsive)
    if r >= 3 and f >= 4:
        return 'Loyal Customers'
    
    # Rule 4: At risk valuable (past high spenders/frequent buyers who went silent)
    if r <= 2 and (f >= 4 or m >= 4):
        return 'At Risk Valuable'
    
    # Rule 5: Promising customers (recent buyers with moderate spend/orders)
    if r >= 4:
        return 'Promising Customers'
    
    # Rule 6: Hibernating customers (low recency, low spend, inactive)
    if r <= 2:
        return 'Hibernating'
    
    # Rule 7: Developing customers (all remaining middle-tier accounts)
    return 'Developing Customers'

# Apply rules
customer_df['Segment'] = customer_df.apply(assign_rfm_segment, axis=1)

print('=== SEGMENT VALIDATION CHECK ===')
print(f'Total Segmented Customers: {len(customer_df)}')
print(f'Unassigned Customers:     {customer_df["Segment"].isnull().sum()} (Must be 0)')
print('\nCustomer Counts per Segment:')
print(customer_df['Segment'].value_counts())


## 🎨 Step 5: Profile & Visualize the 7 Customer Personas
---
> 🧠 **ANALYSIS TYPE**: Multi-Metric Business Profiling & Revenue Contribution  
> 🎯 **WHAT THIS CODE ACTUALLY DOES**:  
> 1. Groups by `Segment` to calculate: Customer Count, % of Customer Base, Total Revenue (£), Revenue Share (%), Median Recency (days), Median Frequency (orders), and Median Spend (£).  
> 2. Plots two comparative bar charts: Customer Volume Share (%) vs. Revenue Contribution (%).  
> 🔍 **WHAT TO OBSERVE**: Look at the **Champions**: they make up only ~12% of the customer base, yet generate over **55% of all company revenue**! Look at **At Risk Valuable**: past VIPs who haven't ordered in months.


In [ ]:
# Aggregate segment profiles in original business units
segment_profile = customer_df.groupby('Segment').agg(
    CustomerCount=('CustomerID', 'count'),
    TotalRevenue=('Monetary', 'sum'),
    MedianRecency=('Recency', 'median'),
    MedianFrequency=('Frequency', 'median'),
    MedianMonetary=('Monetary', 'median'),
    MedianAOV=('AvgOrderValue', 'median')
).reset_index()

total_rev = customer_df['Monetary'].sum()
total_cust = len(customer_df)

segment_profile['CustomerSharePct'] = (segment_profile['CustomerCount'] / total_cust) * 100
segment_profile['RevenueSharePct'] = (segment_profile['TotalRevenue'] / total_rev) * 100
segment_profile = segment_profile.sort_values(by='TotalRevenue', ascending=False)

# Visual comparison: Customer Count vs. Revenue Contribution
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.barplot(data=segment_profile, y='Segment', x='CustomerSharePct', palette='Blues_r', ax=axes[0])
axes[0].set_title('Customer Base Breakdown (% of Customers)', fontweight='bold')
axes[0].set_xlabel('% of Total Customers')
axes[0].set_ylabel('')

sns.barplot(data=segment_profile, y='Segment', x='RevenueSharePct', palette='Greens_r', ax=axes[1])
axes[1].set_title('Revenue Contribution Breakdown (% of Total Spend)', fontweight='bold')
axes[1].set_xlabel('% of Total Gross Revenue')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

display(segment_profile[['Segment', 'CustomerCount', 'CustomerSharePct', 'TotalRevenue', 'RevenueSharePct', 
                         'MedianRecency', 'MedianFrequency', 'MedianMonetary', 'MedianAOV']].round(1))


## 🔬 Step 6: Mathematical ML Preparation (Log-Transform & Scaling)
---
> 🧠 **ANALYSIS TYPE**: Feature Normalization & Skewness Rectification  
> 🎯 **WHAT THIS CODE ACTUALLY DOES**:  
> 1. Isolates continuous RFM features: `['Recency', 'Frequency', 'Monetary']`.  
> 2. Applies natural log transformation `np.log1p()` to compress heavy positive skewness (since K-Means uses Euclidean distance, extreme outliers would otherwise distort cluster centroids).  
> 3. Applies `StandardScaler()` to bring features to zero mean and unit variance.  
> 4. Plots before-and-after distribution histograms.  
> 🔍 **WHAT TO OBSERVE**: Watch how heavily right-skewed variables transform into bell-shaped (Gaussian-like) distributions ready for K-Means.


In [ ]:
# ML Feature Selection
rfm_features = ['Recency', 'Frequency', 'Monetary']

# 1. Apply log1p transformation to fix extreme positive skew
rfm_log = np.log1p(customer_df[rfm_features])

# 2. Standardize features to mean=0, std=1
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)
rfm_scaled_df = pd.DataFrame(rfm_scaled, columns=rfm_features)

# Visual comparison: Before vs After Transformation
fig, axes = plt.subplots(2, 3, figsize=(18, 8))

for i, col in enumerate(rfm_features):
    # Before
    sns.histplot(customer_df[col], bins=30, kde=True, color='#e74c3c', ax=axes[0, i])
    axes[0, i].set_title(f'Raw {col} (Extreme Skew)', fontweight='bold')
    
    # After
    sns.histplot(rfm_scaled_df[col], bins=30, kde=True, color='#2ecc71', ax=axes[1, i])
    axes[1, i].set_title(f'Scaled Log({col}) (Normalized)', fontweight='bold')

plt.tight_layout()
plt.show()


## 📈 Step 7: K-Means Hyperparameter Optimization (Elbow & Silhouette)
---
> 🧠 **ANALYSIS TYPE**: Unsupervised Clustering Optimization ($k$-Selection)  
> 🎯 **WHAT THIS CODE ACTUALLY DOES**:  
> 1. Iterates through candidate cluster counts $k \in [2, 8]$.  
> 2. Computes **Inertia / WCSS** (Within-Cluster Sum of Squares) for the **Elbow Method**.  
> 3. Computes the **Silhouette Score** (measuring how compact clusters are relative to neighboring clusters).  
> 4. Plots dual optimization charts.  
> 🔍 **WHAT TO OBSERVE**: Identify the "elbow" bend at $k=4$ and check the Silhouette peak ($>0.30$). $k=4$ provides the optimal mathematical balance between cluster separation and business interpretability.


In [ ]:
# Evaluate candidate cluster counts from k=2 to k=8
k_range = range(2, 9)
inertia_scores = []
silhouette_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(rfm_scaled)
    inertia_scores.append(km.inertia_)
    silhouette_scores.append(silhouette_score(rfm_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Elbow Method (Inertia)
axes[0].plot(k_range, inertia_scores, marker='o', color='#2980b9', linewidth=2.5)
axes[0].set_title('Elbow Method: Inertia vs. Cluster Count (k)', fontweight='bold')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Within-Cluster Sum of Squares (Inertia)')

# Silhouette Score
axes[1].plot(k_range, silhouette_scores, marker='s', color='#27ae60', linewidth=2.5)
axes[1].set_title('Silhouette Analysis: Separation vs. k', fontweight='bold')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Coefficient')

plt.tight_layout()
plt.show()

for k, s in zip(k_range, silhouette_scores):
    print(f'k = {k} | Silhouette Score: {s:.4f}')


## 🌌 Step 8: Train Winning K-Means Model & 2D PCA Cluster Projection
---
> 🧠 **ANALYSIS TYPE**: Model Training & Dimensionality Reduction (PCA)  
> 🎯 **WHAT THIS CODE ACTUALLY DOES**:  
> 1. Fits the selected K-Means model with $k=4$.  
> 2. Assigns `Cluster_ID` (0, 1, 2, 3) to each customer.  
> 3. Applies **Principal Component Analysis (PCA)** to project the 3D scaled feature space into 2 orthogonal principal components ($PC_1, PC_2$).  
> 4. Generates a 2D scatter plot visualizing cluster boundaries and separation.  
> 🔍 **WHAT TO OBSERVE**: Watch how the 4 clusters occupy distinct geometric territories on the 2D PCA plane.


In [ ]:
# Train winning K-Means model (k=4)
optimal_k = 4
kmeans_model = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
customer_df['Cluster_ID'] = kmeans_model.fit_predict(rfm_scaled)

# Apply PCA for 2D visualization
pca = PCA(n_components=2, random_state=42)
pca_transformed = pca.fit_transform(rfm_scaled)
customer_df['PCA1'] = pca_transformed[:, 0]
customer_df['PCA2'] = pca_transformed[:, 1]

print(f'Explained Variance Ratio by 2 Components: {pca.explained_variance_ratio_.sum()*100:.2f}%')

# 2D PCA Cluster Scatter Plot
plt.figure(figsize=(10, 6))
sns.scatterplot(data=customer_df, x='PCA1', y='PCA2', hue='Cluster_ID', palette='Set1', s=45, alpha=0.85)
plt.title('2D PCA Projection of K-Means Customer Clusters (k=4)', fontsize=14)
plt.xlabel(f'Principal Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}% Variance)', fontweight='bold')
plt.ylabel(f'Principal Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}% Variance)', fontweight='bold')
plt.legend(title='K-Means Cluster', loc='upper right')
plt.tight_layout()
plt.show()

# Profile K-Means clusters in original business units
kmeans_profile = customer_df.groupby('Cluster_ID').agg(
    CustomerCount=('CustomerID', 'count'),
    MedianRecency=('Recency', 'median'),
    MedianFrequency=('Frequency', 'median'),
    MedianMonetary=('Monetary', 'median'),
    TotalRevenue=('Monetary', 'sum')
).reset_index()

kmeans_profile['RevenueSharePct'] = (kmeans_profile['TotalRevenue'] / customer_df['Monetary'].sum()) * 100
display(kmeans_profile.round(1))


## ⚖️ Step 9: Rule-Based Segments vs. K-Means Clusters Comparison
---
> 🧠 **ANALYSIS TYPE**: Cross-Methodology Model Validation & Cross-Tabulation  
> 🎯 **WHAT THIS CODE ACTUALLY DOES**:  
> 1. Builds a cross-tabulation matrix (`pd.crosstab`) mapping the **7 Business Rule Segments** against the **4 K-Means Clusters**.  
> 2. Displays a heatmap showing cluster overlap and mapping.  
> 🔍 **WHAT TO OBSERVE**:  
> - Notice how K-Means Cluster 2 maps almost entirely to **Champions & Loyal Customers** (high frequency, high spend).  
> - Notice how K-Means Cluster 0 captures **Hibernating & At Risk** accounts (high recency).  
> - This proves that unsupervised machine learning independently discovered the exact same customer behavioral boundaries defined by our domain rules!


In [ ]:
# Cross-tabulation heatmap: Rule Segments vs K-Means Clusters
cross_tab = pd.crosstab(customer_df['Segment'], customer_df['Cluster_ID'], margins=False)

plt.figure(figsize=(12, 6))
sns.heatmap(cross_tab, annot=True, fmt='d', cmap='YlGnBu', cbar_kws={'label': 'Customer Count'})
plt.title('Cross-Tabulation: Rule-Based Segments vs. K-Means Clusters', fontsize=14)
plt.xlabel('K-Means Cluster ID (k=4)', fontweight='bold')
plt.ylabel('7-Tier Hierarchical Rule Segment', fontweight='bold')
plt.tight_layout()
plt.show()

display(cross_tab)


## 💾 Step 10: Export Segmented Customer Base Table for Retention
---
> 🧠 **ANALYSIS TYPE**: Analytical Data Pipeline Export  
> 🎯 **WHAT THIS CODE ACTUALLY DOES**:  
> 1. Selects all customer profile features, scores, rule-based segment labels, and K-Means cluster IDs.  
> 2. Exports the dataset to `notebook/data/customer_segmented_data.csv`.  
> 🔍 **WHAT TO OBSERVE**: The file is ready to be consumed directly by **Notebook 3: Cohort Retention & Inactivity Prediction**!


In [ ]:
# Export final analytical table
export_cols = ['CustomerID', 'Recency', 'Frequency', 'Monetary', 'AvgOrderValue', 
               'CustomerTenureDays', 'DistinctSKUs', 'R_Score', 'F_Score', 'M_Score', 
               'Segment', 'Cluster_ID']

export_path = 'data/customer_segmented_data.csv'
customer_df[export_cols].to_csv(export_path, index=False)

print(f'Segmented Customer Dataset successfully saved to: {export_path}')
print(f'Total Profiles Exported: {len(customer_df):,}')
display(customer_df[export_cols].head())
